# MaterialMind-ECE — Phase 7: ECE Material Recommendation Engine
### Unit 3: Machine Learning / AI | Project 7: Electronic Material Clustering

> **MANDATORY SCIENTIFIC DISCLAIMER**  
> **HEURISTIC ECE SCREENING — NOT EXPERIMENTAL VALIDATION.**  
> This recommendation engine provides heuristic material-level screening across the 1,056-material database using available DFT ground-state physical descriptors. It does **NOT** predict device-level properties such as breakdown voltage, loss tangent, mobility, or lifetime.

#### Core Application Profiles:
1. **`POWER_ELECTRONICS`**: Higher band gap is used as an **electronic robustness screening criterion. It is not a prediction of breakdown voltage or critical breakdown field**, with moderate dielectric response and structural density.
2. **`DIELECTRIC_CAPACITIVE`**: High static permittivity ($\varepsilon_r$), high lattice ionic polarization ($f_{\text{ionic}}$), and solid ceramic density.
3. **`RF_HIGH_FREQUENCY`**: A higher band gap is used as a **heuristic insulating-character screening criterion. The model does not predict RF leakage, dielectric loss, or high-frequency device performance**. Low-to-moderate dielectric permittivity is used only as a heuristic material-level screening criterion.
4. **`OPTOELECTRONIC`**: Evaluates band gap as a **band-gap compatibility screening criterion** ($1.0 - 3.0\text{ eV}$), high optical dielectric response ($\varepsilon_\infty = n^2$), and crystalline density.

In [ ]:
import os
import sys

# Add workspace root to Python path
workspace_root = os.path.abspath('..')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backend.recommendation import MaterialRecommendationEngine, DISCLAIMER_TEXT

print(f"Loaded MaterialRecommendationEngine. Disclaimer: {DISCLAIMER_TEXT}")

## 1. Engine Initialization & Profile Configurations
Let's inspect the transparent weights and documented rationales for all 4 profiles.

In [ ]:
engine = MaterialRecommendationEngine()
profiles = engine.get_profiles()

for p_key, p_info in profiles.items():
    print(f"=== Profile: {p_key} ({p_info['name']}) ===")
    print(f"Description: {p_info['description']}")
    print("Weights & Rationales:")
    for w_name, w_val in p_info['weights'].items():
        print(f"  - {w_name}: {w_val} pts | {p_info['rationales'][w_name]}")
    print()

## 2. Execute Recommendations Across All Profiles
We retrieve Top-5 recommendations for each profile and log individual feature contributions and nearest physical neighbors.

In [ ]:
results = {}
for prof in ['POWER_ELECTRONICS', 'DIELECTRIC_CAPACITIVE', 'RF_HIGH_FREQUENCY', 'OPTOELECTRONIC']:
    res = engine.recommend(prof, top_n=5, include_similarity=True)
    results[prof] = res
    
    print(f"\n{'='*30} {prof} {'='*30}")
    print(f"Cluster distribution: {res['cluster_distribution']}")
    display_cols = ['rank', 'material_id', 'formula', 'cluster', 'recommendation_score', 'band_gap', 'poly_total', 'poly_electronic', 'ionic_polarization_fraction', 'density', 'volume']
    print(res['top_materials'][display_cols].to_string(index=False))
    print("\nExplanations:")
    for _, row in res['top_materials'].iterrows():
        print(f"  #{row['rank']} {row['formula']} ({row['material_id']}): {row['explanation']}")

## 3. Nearest Physical Neighbors for Top Recommendations
Checking whether top recommended materials have close physical analogues in the 6D standardized descriptor space.

In [ ]:
nn_records = []
for prof, res in results.items():
    for mid, nn in res['nearest_neighbors'].items():
        nn_records.append({
            'application_profile': prof,
            'material_id': mid,
            'nearest_neighbor_id': nn['neighbor_id'],
            'nearest_neighbor_formula': nn['neighbor_formula'],
            'nearest_neighbor_cluster': nn['neighbor_cluster'],
            'euclidean_distance': nn['euclidean_distance'],
            'similarity_score': nn['similarity_score']
        })
df_nn = pd.DataFrame(nn_records)
print(df_nn.to_string(index=False))

## 4. Sanity Checks and Attribution Verification
We verify:  
1. Scores are strictly bounded in [0, 100].  
2. No duplicate materials within Top 5.  
3. Individual feature contributions sum exactly to the final recommendation score.

In [ ]:
for prof, res in results.items():
    top_df = res['top_materials']
    assert len(top_df) == 5
    assert top_df['material_id'].nunique() == 5
    assert (top_df['recommendation_score'] >= 0).all() and (top_df['recommendation_score'] <= 100).all()
    
    contrib_cols = [c for c in top_df.columns if c.startswith('contrib_')]
    sums = top_df[contrib_cols].sum(axis=1)
    discrepancy = np.abs(sums - top_df['recommendation_score'])
    assert (discrepancy < 1e-4).all(), f"Sum mismatch in {prof}"
    print(f"[PASS] {prof}: Scores bounded, 0 duplicates, attribution sums match exactly (max diff = {discrepancy.max():.2e})")